<a href="https://colab.research.google.com/github/Tejesh18/vcu-finance-dashboard/blob/main/TS_Finance_Dashboard_Data_Refresh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

print("Authentication successful!")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import io

# Build Drive service
drive_service = build('drive', 'v3', credentials=creds)

# Search for the expenditures file in your Drive folder
results = drive_service.files().list(
    q="name contains 'TS E&G Expenditures' and trashed=false",
    fields="files(id, name)"
).execute()

files = results.get('files', [])
for f in files:
    print(f['name'], '→', f['id'])

In [ ]:
file_id = '1bPBamW_Tui3_ty6FImaqANNTsAy2KYXZ'

request = drive_service.files().get_media(fileId=file_id)
file_content = io.BytesIO(request.execute())

xl = pd.read_excel(file_content, sheet_name=None, header=None)
print("Sheets found:", list(xl.keys()))

In [ ]:
import datetime
import re

fy_map = {
    'FY21-22': 'FY22', '21-22(1)': 'FY22', '22-23': 'FY23',
    'TS E&G FY24': 'FY24', 'TS E&G FY25': 'FY25', 'TS E&G FY26': 'FY26'
}

dept_map = {
    '123156': 'Fiscal & Admin Services', '138102': 'Tech Services Development',
    '138103': 'CIO', '138109': 'TAB Furnishings/Operations',
    '138124': 'Tech Services Maintenance', '138007': 'Student Tech Fee',
    '138008': 'Student Tech Fee Differential', '123099': 'Network Services',
    '123117': 'Network Infrastructure', '123102': 'University Computer Center',
    '138104': 'Administrative Systems', '138114': 'Application Services',
    '138116': 'TS Administration', '123152': 'Information Security',
    '123151': 'Technical Support Services', '138005': 'Learning Systems',
    '138014': 'TSS Administration', '138130': 'TS Strategic Communications',
    '138131': 'Student Tech Fee Differential', '146029': 'Adobe Creative Cloud',
    '146811': 'OnePrint', '123131': 'VCUnet Data',
    '123153': 'Advanced Network Technologies', '123154': 'VCUnet Infrastructure',
    '123155': 'Network Operations Center', '138105': 'Administration BI & Analytics',
    '138119': 'Web Content Management', '123116': 'Media Services Projects',
    '123118': 'Media Engineering & Design', '123157': 'Labs & Classrooms Computing',
    '138002': 'Classroom Support Services', '138004': 'Academic Tech Admin',
    '138013': 'Video Production', '138120': 'Capture', '138009': 'InfoSec Overhead',
    '146405': 'Endpoint Computing', '138010': 'Emergency Communications',
    '138017': 'Desktop Services', '138011': 'fixIT', '123100': 'IT Support Center',
    '138110': 'Building Access', '138001': 'Campus Card Services',
    '140801': 'IT Service Management', 'R38001': 'Canon Student Print',
}

skip_categories = [
    'total budget', 'personal services', 'central services', 'total fy',
    'non-personal services', 'non-persoanl services', 'telecommunications',
    'network services', 'university computer center', 'administrative systems',
    'application services', 'academic technology', 'information security',
    'technical support services', 'ts strategic communications', 'nan'
]

def is_skip(cat):
    return str(cat).strip().lower() in skip_categories

def group_category(cat):
    cat = str(cat).lower()
    if any(x in cat for x in ['salary', 'salaries', 'hourly', 'wages', 'faculty', 'it salaries', 'it/univ']):
        return 'Salaries'
    elif 'fringe' in cat:
        return 'Fringes'
    elif any(x in cat for x in ['bonus', 'on call', 'oncall', 'ot/', '/ot', 'vsdp', 'overtime']):
        return 'Bonus/OT/VSDP'
    elif any(x in cat for x in ['software', 'maintenance', 'cloud', 'maint']):
        return 'Software & Maintenance'
    elif any(x in cat for x in ['training', 'travel', 'membership']):
        return 'Training & Travel'
    elif any(x in cat for x in ['supplies', 'postage', 'printing', 'shipping']):
        return 'Supplies & Printing'
    elif any(x in cat for x in ['telecomm', 'telephone', 'network', 'tele ', 'wireless', 'windstream', 'segra', 'level3', 'comcast']):
        return 'Telecomm & Network'
    elif any(x in cat for x in ['recovery', 'recoveries', 'revenue', 'internal charge', 'internal serv', 'sla']):
        return 'Recoveries & Revenue'
    elif any(x in cat for x in ['equipment', 'hardware', 'computer', 'ramtech', 'laptop', 'computing', 'electronic']):
        return 'Equipment & Hardware'
    elif any(x in cat for x in ['contractual', 'consulting', 'technical services', 'skilled services']):
        return 'Contractual & Services'
    else:
        return 'Other'

def parse_month(m):
    try:
        return pd.to_datetime(m.replace('a/o', '').strip(), format='%m/%d/%y')
    except:
        return None

all_rows = []

for sheet_name, df in xl.items():
    fy = fy_map[sheet_name]

    # Only process last 3 years
    if fy not in ['FY24', 'FY25', 'FY26']:
        continue

    month_cols = []
    for row_idx in range(len(df)):
        for col_idx in range(len(df.columns)):
            val = str(df.iloc[row_idx, col_idx])
            if 'a/o' in val and re.search(r'\d+/\d+/\d+', val):
                if col_idx not in [m[0] for m in month_cols]:
                    month_cols.append((col_idx, val.strip()))

    current_dept = 'Unknown'

    for row_idx in range(len(df)):
        for check_col in [0, 1]:
            cell_val = str(df.iloc[row_idx, check_col]).strip()
            if cell_val in dept_map:
                current_dept = dept_map[cell_val]
                break

        for month_col, month_label in month_cols:
            category = None
            for cat_offset in [-3, -2, -1]:
                cat_col = month_col + cat_offset
                if cat_col < 0:
                    continue
                potential_cat = df.iloc[row_idx, cat_col]
                if not pd.isna(potential_cat) and str(potential_cat).strip() not in ['', 'nan']:
                    try:
                        float(potential_cat)
                    except:
                        category = str(potential_cat).strip()
                        break

            if category is None or is_skip(category):
                continue

            try:
                expenditure = float(df.iloc[row_idx, month_col])
                perm_budget = float(df.iloc[row_idx, month_col - 2]) if month_col >= 2 else 0
                current_budget = float(df.iloc[row_idx, month_col - 1]) if month_col >= 1 else 0
                balance = float(df.iloc[row_idx, month_col + 3])
            except:
                continue

            if expenditure == 0:
                continue

            all_rows.append({
                'FY': fy, 'Department': current_dept, 'Month': month_label,
                'Category': category, 'Category_Group': group_category(category),
                'Perm_Budget': perm_budget, 'Current_Budget': current_budget,
                'Expenditure': expenditure, 'Balance': balance
            })

exp_full = pd.DataFrame(all_rows)
exp_full['Date'] = exp_full['Month'].apply(parse_month)
exp_full = exp_full.dropna(subset=['Date'])

# Fiscal month numbering
def get_fiscal_month(date):
    m = date.month
    return m - 6 if m >= 7 else m + 6

month_names = {1:'Jul',2:'Aug',3:'Sep',4:'Oct',5:'Nov',6:'Dec',
               7:'Jan',8:'Feb',9:'Mar',10:'Apr',11:'May',12:'Jun'}

exp_full['Fiscal_Month'] = exp_full['Date'].apply(get_fiscal_month)
exp_full['Month_Name'] = exp_full['Fiscal_Month'].map(month_names)
exp_full['Date'] = exp_full['Date'].dt.strftime('%Y-%m-%d')

# Last month only version for budget comparison
last_month = exp_full.groupby(['FY', 'Department'])['Date'].max().reset_index()
last_month.columns = ['FY', 'Department', 'Last_Date']
exp_last = exp_full.merge(last_month, on=['FY', 'Department'])
exp_last = exp_last[exp_last['Date'] == exp_last['Last_Date']].drop(columns=['Last_Date'])

print(f"Monthly data: {len(exp_full)} rows")
print(f"Last month data: {len(exp_last)} rows")
print("Done!")

In [ ]:
import googleapiclient.http
import os

# Get the folder ID from your Drive folder
results = drive_service.files().list(
    q="name='TS Finance Dashboard - Data Sources' and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields="files(id, name)"
).execute()

folder_id = results.get('files', [])[0]['id']
print(f"Folder found: {folder_id}")

def save_to_drive(df, filename, folder_id):
    # Save locally first
    df.to_csv(filename, index=False)

    # Check if file already exists in Drive
    existing = drive_service.files().list(
        q=f"name='{filename}' and '{folder_id}' in parents and trashed=false",
        fields="files(id)"
    ).execute().get('files', [])

    media = googleapiclient.http.MediaFileUpload(filename, mimetype='text/csv')

    if existing:
        # Update existing file
        drive_service.files().update(fileId=existing[0]['id'], media_body=media).execute()
        print(f"Updated: {filename}")
    else:
        # Create new file
        file_metadata = {'name': filename, 'parents': [folder_id]}
        drive_service.files().create(body=file_metadata, media_body=media).execute()
        print(f"Created: {filename}")

# Save both files
save_to_drive(exp_full, 'expenditures_monthly.csv', folder_id)
save_to_drive(exp_last, 'expenditures_clean.csv', folder_id)

print("\nAll files saved to Google Drive!")

In [ ]:
import gspread

gc = gspread.authorize(creds)

# Open Tomekia's contracts sheet
sheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1F9gtr5UnzdwNHqXJ7kRoQW1jUSCd-e-lJiSjFpw_Iw8/edit')

# List all sheets
worksheets = sheet.worksheets()
print("Sheets found:")
for ws in worksheets:
    print(f"  - {ws.title}")

In [ ]:
import datetime
from dateutil import parser as dateparser

dept_sheets = ['InfoSec/EndPoint', 'Application Svcs', 'TS Administration',
               'Technology Support Svcs', 'Academic Technologies', 'Network/UCC',
               'Administrative Systems', 'Central Maintenance']

all_contracts = []
today = datetime.date.today()

def get_status(renewal):
    if not renewal:
        return 'Unknown'
    try:
        renewal_date = datetime.datetime.strptime(str(renewal), '%Y-%m-%d').date()
        days = (renewal_date - today).days
        if days < 0:
            return 'Expired'
        elif days <= 90:
            return 'Expiring Soon'
        else:
            return 'Active'
    except:
        return 'Unknown'

def days_until(renewal):
    if not renewal:
        return None
    try:
        renewal_date = datetime.datetime.strptime(str(renewal), '%Y-%m-%d').date()
        return (renewal_date - today).days
    except:
        return None

for sheet_name in dept_sheets:
    try:
        ws = sheet.worksheet(sheet_name)
        data = ws.get_all_values()
        if len(data) < 2:
            continue
        for row in data[1:]:
            if len(row) < 4 or not row[0] or row[0].strip() == '':
                continue
            try:
                cost_str = str(row[3]).replace('$','').replace(',','').strip()
                cost = float(cost_str) if cost_str else 0
                if cost <= 0:
                    continue
            except:
                continue
            renewal = row[4] if len(row) > 4 else ''
            try:
                renewal_parsed = dateparser.parse(str(renewal)).strftime('%Y-%m-%d') if renewal else ''
            except:
                renewal_parsed = ''
            all_contracts.append({
                'Department': sheet_name,
                'Vendor': str(row[0]).strip(),
                'Product': str(row[1]).strip() if len(row) > 1 else '',
                'Annual_Cost': cost,
                'Renewal_Date': renewal_parsed,
                'Funding_Source': str(row[5]).strip() if len(row) > 5 else '',
                'Status': get_status(renewal_parsed),
                'Days_Until_Expiry': days_until(renewal_parsed)
            })
    except Exception as e:
        print("Error on " + sheet_name + ": " + str(e))

contracts_df = pd.DataFrame(all_contracts)
junk = ['NOTES:', 'Total Cost:', 'MARIA', 'Facility Support', '']
contracts_df = contracts_df[~contracts_df['Vendor'].isin(junk)]
contracts_df = contracts_df[contracts_df['Annual_Cost'] > 0]

print("Total contracts: " + str(len(contracts_df)))
print(contracts_df['Status'].value_counts())

save_to_drive(contracts_df, 'contracts_clean.csv', folder_id)
print("Contracts saved to Google Drive!")

In [ ]:
sheet2 = gc.open_by_url('https://docs.google.com/spreadsheets/d/1Gb9Lo0DHTHcsHNoKFOWdw-kcjQWH-SSDk0wFmChplXM/edit')

worksheets2 = sheet2.worksheets()
print("Sheets found:")
for ws in worksheets2:
    print(f"  - {ws.title}")

In [ ]:
ws2 = sheet2.worksheet('1-23102')
data2 = ws2.get_all_values()
print("Rows:", len(data2))
for row in data2[:10]:
    print(row)

In [ ]:
# Map cost center codes to department names
po_dept_map = {
    '1-23102': 'University Computer Center',
    '1-23151': 'Technical Support Services',
    '1-23152': 'Information Security',
    '1-23156': 'Fiscal & Admin Services',
    '1-38001': 'Campus Card Services',
    '1-38005': 'Learning Systems',
    '1-38007': 'Student Tech Fee',
    '1-38009': 'InfoSec Overhead',
    '1-38102': 'Tech Services Development',
    '1-38103': 'CIO',
    '1-38104': 'Administrative Systems',
    '1-38105': 'Administration BI & Analytics',
    '1-38109': 'TAB Furnishings/Operations',
    '1-38010': 'Emergency Communications',
    '1-38011': 'fixIT',
    '1-38014': 'TSS Administration',
    '1-38017': 'Desktop Services',
    '1-38114': 'Application Services',
    '1-38124': 'Tech Services Maintenance',
    '1-38131': 'Student Tech Fee Differential',
    '1-38116': 'TS Administration',
    '1-40801': 'IT Service Management',
    '1-46029': 'Adobe Creative Cloud',
    '1-46405': 'Endpoint Computing',
    '2-91833': 'Other',
    '3-86106': 'Telecomm',
    'R-38001': 'Canon Student Print',
}

all_pos = []

for ws_name in [ws.title for ws in sheet2.worksheets()]:
    dept = po_dept_map.get(ws_name.strip(), ws_name.strip())
    try:
        ws2 = sheet2.worksheet(ws_name)
        data2 = ws2.get_all_values()

        # Find header row
        header_row = None
        for i, row in enumerate(data2):
            if 'Date' in row and 'Vendor' in row:
                header_row = i
                break

        if header_row is None:
            continue

        headers = data2[header_row]

        for row in data2[header_row + 1:]:
            if not row[0] or row[0].strip() == '':
                continue
            try:
                amount_str = str(row[4]).replace('$','').replace(',','').strip()
                amount = float(amount_str) if amount_str else 0
                if amount == 0:
                    continue

                balance_str = str(row[6]).replace('$','').replace(',','').strip()
                balance = float(balance_str) if balance_str else 0

                received_str = str(row[5]).replace('$','').replace(',','').strip()
                received = float(received_str) if received_str else 0

                all_pos.append({
                    'Department': dept,
                    'Date': str(row[0]).strip(),
                    'PR_Number': str(row[1]).strip(),
                    'PO_Number': str(row[2]).strip(),
                    'Vendor': str(row[3]).strip(),
                    'Amount': amount,
                    'Amount_Received': received,
                    'Ending_Balance': balance,
                    'Receiver_Date': str(row[7]).strip() if len(row) > 7 else '',
                    'Budget_Code': str(row[8]).strip() if len(row) > 8 else '',
                    'Account_Code': str(row[9]).strip() if len(row) > 9 else '',
                    'Comment': str(row[10]).strip() if len(row) > 10 else '',
                })
            except:
                continue
    except Exception as e:
        print("Error on " + ws_name + ": " + str(e))

po_df = pd.DataFrame(all_pos)
po_df = po_df[po_df['Vendor'].str.strip() != '']

print("Total POs: " + str(len(po_df)))
print(po_df['Department'].value_counts().head(10))

save_to_drive(po_df, 'purchase_orders_clean.csv', folder_id)
print("POs saved to Google Drive!")

In [ ]:
sheet3 = gc.open_by_url('https://docs.google.com/spreadsheets/d/1rRUBJP4EKW9MAeXyKwUNo8j98_0cMI-0StdVd-inE50/edit')

worksheets3 = sheet3.worksheets()
print("Sheets found:")
for ws in worksheets3:
    print(f"  - {ws.title}")

# Preview first sheet
ws3 = sheet3.worksheets()[0]
data3 = ws3.get_all_values()
print("\nFirst sheet preview:")
for row in data3[:8]:
    print(row)

In [ ]:
telecom_sheets = ['FY2027', 'FY2026', 'FY2025', 'FY2024']  # last 3-4 years only

all_telecom = []

for sheet_name in telecom_sheets:
    try:
        ws3 = sheet3.worksheet(sheet_name)
        data3 = ws3.get_all_values()

        # Find header row
        header_row = None
        for i, row in enumerate(data3):
            if 'DATE' in row or 'Date' in row:
                header_row = i
                break

        if header_row is None:
            continue

        for row in data3[header_row + 1:]:
            if not row[0] or row[0].strip() == '':
                continue
            try:
                amount_str = str(row[5]).replace('$','').replace(',','').strip()
                amount = float(amount_str) if amount_str else 0
                if amount == 0:
                    continue

                balance_str = str(row[7]).replace('$','').replace(',','').strip()
                balance = float(balance_str) if balance_str else 0

                received_str = str(row[6]).replace('$','').replace(',','').strip()
                received = float(received_str) if received_str else 0

                all_telecom.append({
                    'FY': sheet_name,
                    'Department': 'Telecommunications',
                    'Date': str(row[0]).strip(),
                    'PR_Number': str(row[1]).strip(),
                    'PO_Number': str(row[2]).strip(),
                    'Contract_Number': str(row[3]).strip(),
                    'Vendor': str(row[4]).strip(),
                    'Amount': amount,
                    'Amount_Received': received,
                    'Ending_Balance': balance,
                    'Receiver_Date': str(row[8]).strip() if len(row) > 8 else '',
                    'Budget_Code': str(row[9]).strip() if len(row) > 9 else '',
                    'Comment': str(row[10]).strip() if len(row) > 10 else '',
                })
            except:
                continue
    except Exception as e:
        print("Error on " + sheet_name + ": " + str(e))

telecom_df = pd.DataFrame(all_telecom)
telecom_df = telecom_df[telecom_df['Vendor'].str.strip() != '']

print("Total Telecom POs: " + str(len(telecom_df)))
print(telecom_df['FY'].value_counts())

save_to_drive(telecom_df, 'telecom_pos_clean.csv', folder_id)
print("Telecom POs saved to Google Drive!")

In [ ]:
sheet4 = gc.open_by_url('https://docs.google.com/spreadsheets/d/1MQryERd37hWGRiJwdZkimcAlbXQuaj4-kmI51eynIjc/edit')

worksheets4 = sheet4.worksheets()
print("Sheets found:")
for ws in worksheets4:
    print(f"  - {ws.title}")

ws4 = sheet4.worksheets()[0]
data4 = ws4.get_all_values()
print("\nFirst sheet preview:")
for row in data4[:8]:
    print(row)

In [ ]:
network_dept_map = {
    '123117': 'Network Infrastructure',
    '123153': 'Advanced Network Technologies',
    '123154': 'VCUnet Infrastructure',
    '123099': 'Network Services',
    '123131': 'VCUnet Data',
    '383100': 'Telecommunications',
    '383101': 'Telecommunications 2',
    '771100': 'Other Network',
}

all_network = []

for ws_name in [ws.title for ws in sheet4.worksheets()]:
    dept = network_dept_map.get(ws_name.strip(), ws_name.strip())
    try:
        ws4 = sheet4.worksheet(ws_name)
        data4 = ws4.get_all_values()

        # Find header row
        header_row = None
        for i, row in enumerate(data4):
            if 'Date' in row or 'DATE' in row:
                header_row = i
                break

        if header_row is None:
            continue

        for row in data4[header_row + 1:]:
            if not row[0] or row[0].strip() == '':
                continue
            try:
                amount_str = str(row[5]).replace('$','').replace(',','').strip()
                amount = float(amount_str) if amount_str else 0
                if amount == 0:
                    continue

                balance_str = str(row[7]).replace('$','').replace(',','').strip()
                balance = float(balance_str) if balance_str else 0

                received_str = str(row[6]).replace('$','').replace(',','').strip()
                received = float(received_str) if received_str else 0

                all_network.append({
                    'Department': dept,
                    'Date': str(row[0]).strip(),
                    'Job_Number': str(row[1]).strip(),
                    'PR_Number': str(row[2]).strip(),
                    'PO_Number': str(row[3]).strip(),
                    'Vendor': str(row[4]).strip(),
                    'Amount': amount,
                    'Amount_Received': received,
                    'Ending_Balance': balance,
                    'Receiver_Date': str(row[8]).strip() if len(row) > 8 else '',
                })
            except:
                continue
    except Exception as e:
        print("Error on " + ws_name + ": " + str(e))

network_df = pd.DataFrame(all_network)
network_df = network_df[network_df['Vendor'].str.strip() != '']

print("Total Network POs: " + str(len(network_df)))
print(network_df['Department'].value_counts())

save_to_drive(network_df, 'network_pos_clean.csv', folder_id)
print("Network POs saved to Google Drive!")

In [ ]:
sheet5 = gc.open_by_url('https://docs.google.com/spreadsheets/d/1VpswBHv8Zx2g-xLY9K0lU9HoQsquuuXquIVu5VlekEk/edit')

worksheets5 = sheet5.worksheets()
print("Sheets found:")
for ws in worksheets5:
    print(f"  - {ws.title}")

ws5 = sheet5.worksheets()[0]
data5 = ws5.get_all_values()
print("\nFirst sheet preview:")
for row in data5[:8]:
    print(row)

In [ ]:
jv_sheets = ['FY2027', 'FY2026', 'FY2025', 'FY2024']

all_jv = []

for sheet_name in jv_sheets:
    try:
        ws5 = sheet5.worksheet(sheet_name)
        data5 = ws5.get_all_values()

        # Find header row
        header_row = None
        for i, row in enumerate(data5):
            if 'Date' in row or 'DATE' in row:
                header_row = i
                break

        if header_row is None:
            continue

        for row in data5[header_row + 1:]:
            if not row[0] or row[0].strip() == '':
                continue
            try:
                amount_str = str(row[3]).replace('$','').replace(',','').strip()
                amount = float(amount_str) if amount_str else 0
                if amount == 0:
                    continue

                all_jv.append({
                    'FY': sheet_name,
                    'Date': str(row[0]).strip(),
                    'HD_Number': str(row[1]).strip(),
                    'Department': str(row[2]).strip(),
                    'Amount': amount,
                    'JV_Number': str(row[4]).strip() if len(row) > 4 else '',
                    'Budget_Code': str(row[5]).strip() if len(row) > 5 else '',
                    'Comment': str(row[6]).strip() if len(row) > 6 else '',
                })
            except:
                continue
    except Exception as e:
        print("Error on " + sheet_name + ": " + str(e))

jv_df = pd.DataFrame(all_jv)
jv_df = jv_df[jv_df['Department'].str.strip() != '']

print("Total JV entries: " + str(len(jv_df)))
print(jv_df['FY'].value_counts())

save_to_drive(jv_df, 'jv_log_clean.csv', folder_id)
print("JV Log saved to Google Drive!")

In [ ]:
import pandas as pd
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from google.auth import default
import io

creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)

# Find the clean file
results = drive_service.files().list(
    q="name='expenditures_clean.csv' and trashed=false",
    fields="files(id, name)"
).execute()
print(results.get('files', []))

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from google.auth import default
import pandas as pd
import io
import googleapiclient.http

creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)

# Read clean file
file_id = '1v1a-IlRhE9446ZNICFodfZySOdukrP5U'
request = drive_service.files().get_media(fileId=file_id)
file_content = io.BytesIO(request.execute())
df = pd.read_csv(file_content)

# Group departments into divisions
division_map = {
    'Fiscal & Admin Services': 'Central Services',
    'CIO': 'Central Services',
    'Tech Services Development': 'Central Services',
    'Tech Services Maintenance': 'Central Services',
    'TAB Furnishings/Operations': 'Central Services',
    'TS Strategic Communications': 'Central Services',
    'IT Service Management': 'Central Services',
    'Canon Student Print': 'Central Services',
    'Student Tech Fee': 'Central Services',
    'Student Tech Fee Differential': 'Central Services',
    'Adobe Creative Cloud': 'Central Services',
    'Network Services': 'Network & Infrastructure',
    'Network Infrastructure': 'Network & Infrastructure',
    'Network Operations Center': 'Network & Infrastructure',
    'VCUnet Data': 'Network & Infrastructure',
    'VCUnet Infrastructure': 'Network & Infrastructure',
    'Advanced Network Technologies': 'Network & Infrastructure',
    'OnePrint': 'Network & Infrastructure',
    'University Computer Center': 'Technical Support Services',
    'Technical Support Services': 'Technical Support Services',
    'TSS Administration': 'Technical Support Services',
    'Desktop Services': 'Technical Support Services',
    'fixIT': 'Technical Support Services',
    'IT Support Center': 'Technical Support Services',
    'Building Access': 'Technical Support Services',
    'Campus Card Services': 'Technical Support Services',
    'Emergency Communications': 'Technical Support Services',
    'Application Services': 'Application Services',
    'TS Administration': 'Application Services',
    'Web Content Management': 'Application Services',
    'Administration BI & Analytics': 'Application Services',
    'Administrative Systems': 'Application Services',
    'Information Security': 'Information Security',
    'InfoSec Overhead': 'Information Security',
    'Endpoint Computing': 'Information Security',
    'Learning Systems': 'Academic Technology',
    'Academic Tech Admin': 'Academic Technology',
    'Classroom Support Services': 'Academic Technology',
    'Labs & Classrooms Computing': 'Academic Technology',
    'Media Engineering & Design': 'Academic Technology',
    'Media Services Projects': 'Academic Technology',
    'Video Production': 'Academic Technology',
    'Capture': 'Academic Technology',
}

df['Division'] = df['Department'].map(division_map).fillna('Other')
print(df['Division'].value_counts())

# Save back to Drive
df.to_csv('expenditures_clean.csv', index=False)
media = googleapiclient.http.MediaFileUpload('expenditures_clean.csv', mimetype='text/csv')
drive_service.files().update(fileId=file_id, media_body=media).execute()
print("Saved to Google Drive!")

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from google.auth import default
import pandas as pd
import io
import gspread

creds, _ = default()
gc = gspread.authorize(creds)

sheet3 = gc.open_by_url('https://docs.google.com/spreadsheets/d/1rRUBJP4EKW9MAeXyKwUNo8j98_0cMI-0StdVd-inE50/edit')

# Check FY2026 structure
ws = sheet3.worksheet('FY2026')
data = ws.get_all_values()
print("Header:", data[1])
print("Row 2:", data[2])
print("Row 3:", data[3])
print("Row 4:", data[4])

In [ ]:
import datetime
import googleapiclient.http

drive_service = build('drive', 'v3', credentials=creds)

telecom_sheets_config = {
    'FY2027': {'date': 0, 'pr': 1, 'po': 2, 'contract': 3, 'vendor': 4, 'amount': 5, 'received': 6, 'balance': 7, 'receiver': 8, 'budget': 9},
    'FY2026': {'date': 0, 'pr': 1, 'po': None, 'contract': 2, 'vendor': 3, 'amount': 4, 'received': 5, 'balance': 6, 'receiver': 7, 'budget': 8},
    'FY2025': {'date': 0, 'pr': 1, 'po': None, 'contract': 2, 'vendor': 3, 'amount': 4, 'received': 5, 'balance': 6, 'receiver': 7, 'budget': 8},
    'FY2024': {'date': 0, 'pr': 1, 'po': None, 'contract': 2, 'vendor': 3, 'amount': 4, 'received': 5, 'balance': 6, 'receiver': 7, 'budget': 8},
}

all_telecom = []

for sheet_name, cols in telecom_sheets_config.items():
    try:
        ws = sheet3.worksheet(sheet_name)
        data = ws.get_all_values()

        header_row = None
        for i, row in enumerate(data):
            if 'DATE' in row or 'Date' in row:
                header_row = i
                break
        if header_row is None:
            continue

        for row in data[header_row + 1:]:
            if not row[0] or row[0].strip() == '':
                continue
            try:
                amount_str = str(row[cols['amount']]).replace('$','').replace(',','').strip()
                amount = float(amount_str) if amount_str else 0
                if amount == 0:
                    continue

                vendor = str(row[cols['vendor']]).strip()
                if not vendor:
                    continue

                balance_str = str(row[cols['balance']]).replace('$','').replace(',','').strip()
                balance = float(balance_str) if balance_str else 0

                received_str = str(row[cols['received']]).replace('$','').replace(',','').strip()
                received = float(received_str) if received_str else 0

                all_telecom.append({
                    'FY': sheet_name,
                    'Department': 'Telecommunications',
                    'Date': str(row[cols['date']]).strip(),
                    'PR_Number': str(row[cols['pr']]).strip(),
                    'Vendor': vendor,
                    'Amount': amount,
                    'Amount_Received': received,
                    'Ending_Balance': balance,
                    'Receiver_Date': str(row[cols['receiver']]).strip() if cols['receiver'] < len(row) else '',
                    'Budget_Code': str(row[cols['budget']]).strip() if cols['budget'] < len(row) else '',
                })
            except Exception as e:
                continue
    except Exception as e:
        print(f"Error on {sheet_name}: {e}")

telecom_df = pd.DataFrame(all_telecom)
telecom_df = telecom_df[telecom_df['Vendor'].str.strip() != '']

print(f"Total: {len(telecom_df)}")
print(telecom_df['FY'].value_counts())
print(telecom_df['Vendor'].value_counts().head(10))

In [ ]:
# Find the telecom CSV file ID
results = drive_service.files().list(
    q="name='telecom_pos_clean.csv' and trashed=false",
    fields="files(id, name)"
).execute()
file_id = results.get('files', [])[0]['id']

telecom_df.to_csv('telecom_pos_clean.csv', index=False)
media = googleapiclient.http.MediaFileUpload('telecom_pos_clean.csv', mimetype='text/csv')
drive_service.files().update(fileId=file_id, media_body=media).execute()
print("Saved!")

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from google.auth import default
import gspread
import pandas as pd
import googleapiclient.http
import io

creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

sheet_fy27 = gc.open_by_url('https://docs.google.com/spreadsheets/d/1NE9_S_7Y1csvLmb3ZlNI1JTBnbBlklIglVwXWBNmsLw/edit')
ws = sheet_fy27.worksheets()[0]
data = ws.get_all_values()

print("Headers:", data[0])
print("Row 1:", data[1])
print("Total rows:", len(data))

In [ ]:
for i, row in enumerate(data):
    print(f"Row {i}: {row}")

In [ ]:
all_rows = []
for row in data[7:44]:  # skip headers, stop before Total
    if not row[0] or row[0].strip() == '' or row[0] == 'Total':
        continue
    try:
        budget = float(str(row[1]).replace('$','').replace(',','').strip())
        expenses = float(str(row[2]).replace('$','').replace(',','').strip())
        commitments = float(str(row[3]).replace('$','').replace(',','').strip())
        difference = float(str(row[4]).replace('$','').replace(',','').strip())
        dept = str(row[5]).strip()

        all_rows.append({
            'Index': row[0].strip(),
            'Department': dept,
            'FY': 'FY27',
            'Budget': budget,
            'Expenses': expenses,
            'Commitments': commitments,
            'Difference': difference,
            'As_Of': '2026-07-30',
            'Utilization_Pct': round(expenses / budget * 100, 1) if budget > 0 else 0
        })
    except:
        continue

fy27_df = pd.DataFrame(all_rows)
print(f"Total rows: {len(fy27_df)}")
print(f"Total Budget: ${fy27_df['Budget'].sum():,.0f}")
print(f"Total Expenses: ${fy27_df['Expenses'].sum():,.0f}")

fy27_df.to_csv('fy27_budget.csv', index=False)

# Save to Drive
results = drive_service.files().list(
    q="name='TS Finance Dashboard - Data Sources' and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields="files(id, name)"
).execute()
folder_id = results.get('files', [])[0]['id']

media = googleapiclient.http.MediaFileUpload('fy27_budget.csv', mimetype='text/csv')
file_metadata = {'name': 'fy27_budget.csv', 'parents': [folder_id]}
drive_service.files().create(body=file_metadata, media_body=media).execute()
print("Saved to Drive!")

In [ ]:
# Read expenditures clean to get monthly averages by department
request = drive_service.files().get_media(fileId='1v1a-IlRhE9446ZNICFodfZySOdukrP5U')
file_content = io.BytesIO(request.execute())
exp_df = pd.read_csv(file_content)

# Get average monthly expenditure per department across FY24-26
exp_df['Date'] = pd.to_datetime(exp_df['Date'])
monthly_avg = exp_df.groupby(['Department', 'FY'])['Expenditure'].sum().reset_index()
avg_by_dept = monthly_avg.groupby('Department')['Expenditure'].mean().reset_index()
avg_by_dept.columns = ['Department', 'Avg_Annual_Spend']

# Merge with FY27 data
fy27_df = pd.read_csv('fy27_budget.csv')
fy27_with_proj = fy27_df.merge(avg_by_dept, on='Department', how='left')
fy27_with_proj['Projected_FY27'] = fy27_with_proj['Avg_Annual_Spend']

print(fy27_with_proj[['Department', 'Budget', 'Expenses', 'Projected_FY27']].head(10))

fy27_with_proj.to_csv('fy27_budget.csv', index=False)

# Update in Drive
results = drive_service.files().list(
    q="name='fy27_budget.csv' and trashed=false",
    fields="files(id, name)"
).execute()
file_id = results.get('files', [])[0]['id']
media = googleapiclient.http.MediaFileUpload('fy27_budget.csv', mimetype='text/csv')
drive_service.files().update(fileId=file_id, media_body=media).execute()
print("Updated!")

In [ ]:
# Department name mapping FY27 -> expenditures_clean
dept_name_map = {
    'STF': 'Student Tech Fee',
    'TS Development': 'Tech Services Development',
    'CIO': 'CIO',
    'TS Maintenance': 'Tech Services Maintenance',
    'TAB Furnishings/Operations': 'TAB Furnishings/Operations',
    'Planning & Project Mgmt.': 'Adobe Creative Cloud',
    'Adobe Creative Cloud': 'Adobe Creative Cloud',
    'Network Infrastructure': 'Network Infrastructure',
    'Network Operations': 'Network Operations Center',
    'Network Tech.': 'Advanced Network Technologies',
    'Network Infra. Upgrade': 'VCUnet Infrastructure',
    'Network Data': 'VCUnet Data',
    'Admin. Systems': 'Administrative Systems',
    'Admin. BI & Analytics': 'Administration BI & Analytics',
    'Information Security': 'Information Security',
    'Computing Sppt. Serv.': 'InfoSec Overhead',
    'Endpoint Computing': 'Endpoint Computing',
    'UCC': 'University Computer Center',
    'UCC - NOC': 'Network Operations Center',
    'Applic. Services': 'Application Services',
    'TS Administration': 'TS Administration',
    'Media Sppt. Projects': 'Media Services Projects',
    'Media Eng./Design': 'Media Engineering & Design',
    'Instructional Computing Serv.': 'Labs & Classrooms Computing',
    'Classroom Sppt. Serv.': 'Classroom Support Services',
    'Academic Technologies': 'Academic Tech Admin',
    'Learning Systems': 'Learning Systems',
    'Video/Teleconference': 'Video Production',
    'Emergency Comm.': 'Emergency Communications',
    'TS Serv. Admin.': 'Campus Card Services',
    'Desktop Services': 'Desktop Services',
    'IT Support Ctr.': 'IT Support Center',
    'IT Serv. Mgmt.': 'IT Service Management',
    'fixIT': 'fixIT',
    'Building Access': 'Building Access',
    'Campus Card Serv.': 'Campus Card Services',
    'Canon Student Print': 'Canon Student Print',
    'Fiscal & Admin Services': 'Fiscal & Admin Services',
}

fy27_df['Department_Mapped'] = fy27_df['Department'].map(dept_name_map)
fy27_with_proj = fy27_df.merge(avg_by_dept, left_on='Department_Mapped', right_on='Department', how='left')
fy27_with_proj['Projected_FY27'] = fy27_with_proj['Avg_Annual_Spend']
fy27_with_proj = fy27_with_proj.drop(columns=['Department_y']).rename(columns={'Department_x': 'Department'})

print(fy27_with_proj[['Department', 'Budget', 'Expenses', 'Projected_FY27']].to_string())

fy27_with_proj.to_csv('fy27_budget.csv', index=False)
media = googleapiclient.http.MediaFileUpload('fy27_budget.csv', mimetype='text/csv')
drive_service.files().update(fileId=file_id, media_body=media).execute()
print("Updated!")

In [ ]:
# Fix Canon Student Print and negative budget depts
fy27_with_proj.loc[fy27_with_proj['Budget'] <= 0, 'Projected_FY27'] = None

# Round numbers
fy27_with_proj['Projected_FY27'] = fy27_with_proj['Projected_FY27'].round(2)

print(fy27_with_proj[['Department', 'Budget', 'Expenses', 'Projected_FY27']].to_string())

fy27_with_proj.to_csv('fy27_budget.csv', index=False)
media = googleapiclient.http.MediaFileUpload('fy27_budget.csv', mimetype='text/csv')
drive_service.files().update(fileId=file_id, media_body=media).execute()
print("Done!")

In [ ]:
import pandas as pd
df = pd.read_csv('expenditures_clean.csv')
print(df.groupby('FY')[['Perm_Budget','Expenditure']].sum())

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from google.auth import default
import pandas as pd
import io

creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)

file_id = '1ae7GOecvnJDhcXb9D_gzGPE-CaBtZXEq'
request = drive_service.files().get_media(fileId=file_id)
df = pd.read_csv(io.BytesIO(request.execute()))

print(df.groupby('FY')[['Perm_Budget','Expenditure']].sum())

In [ ]:
# Get last month only per FY per department
df['Date'] = pd.to_datetime(df['Date'])
last_month = df.groupby(['FY','Department'])['Date'].max().reset_index()
last_month.columns = ['FY','Department','Last_Date']
df_last = df.merge(last_month, on=['FY','Department'])
df_last = df_last[df_last['Date'] == df_last['Last_Date']]

print(df_last.groupby('FY')[['Perm_Budget','Expenditure']].sum())

In [ ]:
# Check how many rows per FY per department in last month
check = df_last.groupby(['FY','Department']).size().reset_index(name='row_count')
print(check[check['row_count']>1].head(20))
print("\nTotal departments FY26:", df_last[df_last['FY']=='FY26']['Department'].nunique())
print("\nFY26 budget by dept:")
print(df_last[df_last['FY']=='FY26'].groupby('Department')['Perm_Budget'].sum().sort_values(ascending=False).head(10))

In [ ]:
# Fix: get budget per department as max (not sum across categories)
budget_per_dept = df_last.groupby(['FY','Department'])['Perm_Budget'].max().reset_index()
expenditure_per_dept = df_last.groupby(['FY','Department'])['Expenditure'].sum().reset_index()

summary = budget_per_dept.merge(expenditure_per_dept, on=['FY','Department'])
fy_summary = summary.groupby('FY')[['Perm_Budget','Expenditure']].sum()
print(fy_summary)
print("\nFY26 budget by dept (max):")
print(summary[summary['FY']=='FY26'].sort_values('Perm_Budget',ascending=False).head(10)[['Department','Perm_Budget','Expenditure']])

In [ ]:
import googleapiclient.http

# Remove Canon Student Print and fix budget
summary_clean = summary[summary['Department'] != 'Canon Student Print'].copy()

fy_summary_clean = summary_clean.groupby('FY')[['Perm_Budget','Expenditure']].sum()
print(fy_summary_clean)
print("\nUtilization:")
print((fy_summary_clean['Expenditure']/fy_summary_clean['Perm_Budget']*100).round(1))

In [ ]:
exp_by_fy = summary_clean.groupby('FY')['Expenditure'].sum()
print(exp_by_fy)